In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg
wget --no-verbose --output-document=image_cc_2.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_2.jpg

# Lecture 15 - Multiple View Geometry

Epipolar geometry on two real photos of Christ Church Meadow Building: the fundamental matrix with the eight-point algorithm (and why normalisation and the rank-2 constraint matter), what goes wrong when the scene is (almost) a plane, and a two-view reconstruction by triangulation.

No GPU needed.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
scale = 0.2   # the originals are 6016x4016
image1 = cv2.cvtColor(cv2.resize(cv2.imread('image_cc_1.jpg'), (0, 0), fx=scale, fy=scale, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
image2 = cv2.cvtColor(cv2.resize(cv2.imread('image_cc_2.jpg'), (0, 0), fx=scale, fy=scale, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)

# SIFT matches with the ratio test (Lecture 5)
sift = cv2.SIFT_create(4000)
kp1, des1 = sift.detectAndCompute(cv2.cvtColor(image1, cv2.COLOR_RGB2GRAY), None)
kp2, des2 = sift.detectAndCompute(cv2.cvtColor(image2, cv2.COLOR_RGB2GRAY), None)
matches = [m for m, n in cv2.BFMatcher().knnMatch(des1, des2, k=2) if m.distance < 0.8 * n.distance]
pts1 = np.float64([kp1[m.queryIdx].pt for m in matches])
pts2 = np.float64([kp2[m.trainIdx].pt for m in matches])
print(f'{len(matches)} matches')

fig, axs = plt.subplots(1, 2, figsize=(20, 7))
for ax, im in zip(axs, [image1, image2]):
    ax.imshow(im)
    ax.axis('off')
axs[0].set_title('Image 1')
axs[1].set_title('Image 2')
plt.show()

## Plane and parallax

Most matches lie on the façade, which is (almost) a plane: a single homography maps them from one image to the other. Points *off* the plane (trees, lamp posts, the ground) do not follow the homography. Their residual displacement, the **parallax**, is what tells us about the 3D structure and the epipolar geometry.

We estimate both models robustly. For the fundamental matrix we use MAGSAC, a RANSAC variant (Practical 1) that copes better with this situation.

In [ ]:
H, h_inliers = cv2.findHomography(pts1, pts2, cv2.RANSAC, 2.0)
F_cv, f_inliers = cv2.findFundamentalMat(pts1, pts2, cv2.USAC_MAGSAC, 1.0, 0.999, 10000)
h_inliers, f_inliers = h_inliers.ravel() > 0, f_inliers.ravel() > 0
plane = f_inliers & h_inliers
parallax = f_inliers & ~h_inliers
print(f'homography inliers: {h_inliers.sum()}, fundamental matrix inliers: {f_inliers.sum()}, off the plane: {parallax.sum()}')

# where the homography sends the points of image 1, and how far off the plane points are
transferred = cv2.perspectiveTransform(pts1[:, None], H)[:, 0]
fig, axs = plt.subplots(1, 2, figsize=(20, 7))
axs[0].imshow(image1)
axs[0].scatter(*pts1[plane].T, s=6, c='deepskyblue', label='on the plane (homography inlier)')
axs[0].scatter(*pts1[parallax].T, s=20, c='red', label='off the plane')
axs[0].legend(loc='lower left')
axs[0].set_title('Image 1: matches consistent with the epipolar geometry')
axs[1].imshow(image2)
axs[1].quiver(*transferred[parallax].T, *(pts2[parallax] - transferred[parallax]).T, color='red', angles='xy', scale_units='xy', scale=1, width=0.002)
axs[1].set_xlim(0, image2.shape[1])
axs[1].set_ylim(image2.shape[0], 0)
axs[1].set_title('Image 2: parallax = true position - position predicted by the plane')
for ax in axs:
    ax.axis('off')
savefig('mvg_plane_parallax')
plt.show()

## Epipolar lines

A point $\mathbf{x}_1$ in image 1 must lie on the line $\mathbf{l}_2 = F \mathbf{x}_1$ in image 2 (and $\mathbf{x}_2$ on $\mathbf{l}_1 = F^\top \mathbf{x}_2$ in image 1). All epipolar lines pass through the epipoles, the null vectors of $F$ and $F^\top$.

In [ ]:
def draw_lines(ax, lines, image, colors, linewidth=1):
    w = image.shape[1]
    for (a, b, c), color in zip(lines, colors):
        ax.plot([0, w], [-c / b, -(c + a * w) / b], '-', color=color, linewidth=linewidth)
    ax.set_xlim(0, w)
    ax.set_ylim(image.shape[0], 0)

def epipole(F):
    e = np.linalg.svd(F)[2][-1]
    return e[:2] / e[2]

hom = lambda p: np.c_[p, np.ones(len(p))]
rng = np.random.default_rng(1)
idx = rng.choice(np.where(parallax)[0], 6, replace=False).tolist() + rng.choice(np.where(plane)[0], 6, replace=False).tolist()
colors = plt.cm.tab20(np.linspace(0, 1, len(idx)))
fig, axs = plt.subplots(1, 2, figsize=(20, 7))
axs[0].imshow(image1)
draw_lines(axs[0], hom(pts2[idx]) @ F_cv, image1, colors)
axs[0].scatter(*pts1[idx].T, c=colors, s=60, edgecolors='white', zorder=3)
axs[1].imshow(image2)
draw_lines(axs[1], hom(pts1[idx]) @ F_cv.T, image2, colors)
axs[1].scatter(*pts2[idx].T, c=colors, s=60, edgecolors='white', zorder=3)
for ax, F_ in zip(axs, [F_cv, F_cv.T]):
    ax.set_title(f'epipole at ({epipole(F_)[0]:.0f}, {epipole(F_)[1]:.0f}), outside the image')
    ax.axis('off')
savefig('mvg_epipolar_lines')
plt.show()

## The eight-point algorithm

Every correspondence gives one linear equation $\mathbf{x}_2^\top F \mathbf{x}_1 = 0$ in the 9 entries of $F$. Stack them into $A \mathbf{f} = 0$ and take the right singular vector of $A$ with the smallest singular value. Two details matter in practice:

* **Normalisation** (Hartley): pixel coordinates are in the hundreds, so the entries of $A$ range over many orders of magnitude and the least-squares problem is badly conditioned. Translate and scale the points so that they are centred with an average distance of $\sqrt{2}$, solve, then undo the transformation.
* **Rank 2**: the solution of the linear system is generally not singular, so its epipolar lines do not meet in one point. Set the smallest singular value to zero.

We measure the quality by the *symmetric epipolar distance*: how far (in pixels) each point is from the epipolar line of its match.

In [ ]:
def normalisation(p):
    c = p.mean(0)
    s = np.sqrt(2) / np.linalg.norm(p - c, axis=1).mean()
    return np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])

def eight_point(p1, p2, normalise=True, rank2=True):
    T1, T2 = (normalisation(p1), normalisation(p2)) if normalise else (np.eye(3), np.eye(3))
    x1, x2 = hom(p1) @ T1.T, hom(p2) @ T2.T
    A = np.stack([x2[:, 0] * x1[:, 0], x2[:, 0] * x1[:, 1], x2[:, 0],
                  x2[:, 1] * x1[:, 0], x2[:, 1] * x1[:, 1], x2[:, 1],
                  x1[:, 0], x1[:, 1], np.ones(len(x1))], axis=1)
    F = np.linalg.svd(A)[2][-1].reshape(3, 3)
    if rank2:
        U, S, Vt = np.linalg.svd(F)
        F = U @ np.diag([S[0], S[1], 0]) @ Vt
    F = T2.T @ F @ T1
    return F / np.linalg.norm(F)

def epipolar_distance(F, p1, p2):
    l2, l1 = hom(p1) @ F.T, hom(p2) @ F
    d2 = np.abs((hom(p2) * l2).sum(1)) / np.hypot(l2[:, 0], l2[:, 1])
    d1 = np.abs((hom(p1) * l1).sum(1)) / np.hypot(l1[:, 0], l1[:, 1])
    return (d1 + d2) / 2

# 1) fit on all inliers: with many accurate points every variant works
p1, p2 = pts1[f_inliers], pts2[f_inliers]
for normalise in [False, True]:
    for rank2 in [False, True]:
        d = epipolar_distance(eight_point(p1, p2, normalise, rank2), p1, p2)
        print(f'all {len(p1)} inliers   normalised: {normalise!s:5}  rank 2: {rank2!s:5}  epipolar distance: median {np.median(d):.2f} px')

# 2) the realistic case inside RANSAC: few correspondences with noise. Repeat 200 times with
#    12 random correspondences (half of them off the plane) plus 1 px of noise, evaluate on all inliers.
rng = np.random.default_rng(0)
off, on = np.where(parallax[f_inliers])[0], np.where(plane[f_inliers])[0]
errors = {(n, r): [] for n in [False, True] for r in [False, True]}
for _ in range(200):
    sub = np.concatenate([rng.choice(off, 6, replace=False), rng.choice(on, 6, replace=False)])
    q1, q2 = p1[sub] + rng.normal(0, 1, (12, 2)), p2[sub] + rng.normal(0, 1, (12, 2))
    for (n, r) in errors:
        errors[(n, r)].append(np.median(epipolar_distance(eight_point(q1, q2, n, r), p1, p2)))
plt.figure(figsize=(9, 4))
labels = [f'normalised={n}\nrank 2={r}' for n, r in errors]
plt.boxplot([np.clip(e, 0, 100) for e in errors.values()], showfliers=False)
plt.xticks(range(1, 5), labels)
plt.ylabel('median epipolar distance (px)')
plt.title('Eight-point algorithm from 12 noisy correspondences (200 trials)')
plt.grid(alpha=0.3)
savefig('mvg_eight_point_normalisation')
plt.show()
for (n, r), e in errors.items():
    print(f'normalised: {n!s:5}  rank 2: {r!s:5}  median over trials: {np.median(e):6.2f} px')

In [ ]:
# the rank-2 constraint (Lecture slide "Enforcing rank-2 constraint"): epipolar lines of the
# initial (full-rank) estimate do not meet in a single point; after rank-2 projection they do.
# The effect is easiest to see with only a few, noisy correspondences.
rng = np.random.default_rng(3)
sub = rng.choice(len(p1), 12, replace=False)
noisy1, noisy2 = p1[sub] + rng.normal(0, 2, (12, 2)), p2[sub] + rng.normal(0, 2, (12, 2))
F_rank2 = eight_point(noisy1, noisy2, normalise=True, rank2=True)
e = epipole(F_rank2.T)                                 # the epipole in image 2
shown = hom(p1[rng.choice(len(p1), 15, replace=False)])
fig, axs = plt.subplots(1, 2, figsize=(20, 8))
for ax, rank2 in zip(axs, [False, True]):
    F_ = eight_point(noisy1, noisy2, normalise=True, rank2=rank2)
    for a, b, c in shown @ F_.T:
        xs = np.array([e[0] - 1000, e[0] + 1000])
        ax.plot(xs, -(c + a * xs) / b, '-', linewidth=1)
    ax.plot(*e, 'k+', markersize=20)
    ax.set_xlim(e[0] - 60, e[0] + 60)                  # zoom in around the epipole
    ax.set_ylim(e[1] + 60, e[1] - 60)
    ax.set_aspect('equal')
    ax.set_title(('rank-2 estimate: all lines meet at the epipole' if rank2 else 'initial estimate (rank 3): lines do not meet in one point') + '\n(120x120 px around the epipole of image 2)')
savefig('mvg_rank2')
plt.show()

## Degenerate configuration: a plane

If all points lie on a plane, the correspondences are explained by a homography $H$ and the eight-point system no longer has a unique solution: $F = [\mathbf{e}_2]_\times H$ fits for *any* epipole $\mathbf{e}_2$. We can see this in the singular values of $A$: with points on the plane only, three of them are (close to) zero.

In [ ]:
def design_matrix(p1, p2):
    T1, T2 = normalisation(p1), normalisation(p2)
    x1, x2 = hom(p1) @ T1.T, hom(p2) @ T2.T
    return np.stack([x2[:, 0] * x1[:, 0], x2[:, 0] * x1[:, 1], x2[:, 0], x2[:, 1] * x1[:, 0], x2[:, 1] * x1[:, 1], x2[:, 1], x1[:, 0], x1[:, 1], np.ones(len(x1))], axis=1)

plt.figure(figsize=(8, 4))
for name, mask in [('points on the façade only', plane), ('all inliers (with parallax)', f_inliers)]:
    s = np.linalg.svd(design_matrix(pts1[mask], pts2[mask]))[1]
    plt.semilogy(range(1, 10), s / s[0], 'o-', label=name)
plt.xlabel('index of the singular value of A')
plt.ylabel('relative singular value')
plt.legend()
plt.grid(alpha=0.3)
savefig('mvg_degenerate_plane')
plt.show()

## Two-view reconstruction

With the intrinsics $K$ we get the essential matrix $E = K^\top F K$, which decomposes into the rotation and the (unit-length) translation between the cameras. The focal length is in the photos' EXIF data: 24 mm on a full-frame (36 mm wide) sensor. Then we **triangulate** every correspondence: find the 3D point whose projections are closest to the two observations. The reconstruction is only defined up to scale.

In [ ]:
h, w = image1.shape[:2]
f = 24 / 36 * w
K = np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1]])
E = K.T @ F_cv @ K
# E decomposes into four (R, t) candidates; only one puts the points in front of both cameras
def triangulate(R, t):
    P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = K @ np.hstack([R, t])
    Xh = cv2.triangulatePoints(P1, P2, pts1[f_inliers].T, pts2[f_inliers].T)
    X = (Xh[:3] / Xh[3]).T
    return X, (X[:, 2] > 0) & ((R @ X.T + t)[2] > 0), P1

R1, R2, t0 = cv2.decomposeEssentialMat(E)
candidates = [(R1, t0), (R1, -t0), (R2, t0), (R2, -t0)]
counts = [triangulate(R_, t_)[1].sum() for R_, t_ in candidates]
for (R_, t_), c in zip(candidates, counts):
    print(f'rotation {np.degrees(np.arccos(np.clip((np.trace(R_) - 1) / 2, -1, 1))):5.1f} deg, t = {t_.ravel().round(2)}: {c} points in front of both cameras')
R, t = candidates[int(np.argmax(counts))]
X, in_front, P1 = triangulate(R, t)
reproj = (P1 @ np.c_[X, np.ones(len(X))].T)
reproj = (reproj[:2] / reproj[2]).T
print(f'reprojection error (image 1): median {np.median(np.linalg.norm(reproj - pts1[f_inliers], axis=1)):.2f} px')

colors = image1[pts1[f_inliers][:, 1].astype(int), pts1[f_inliers][:, 0].astype(int)] / 255
on_plane = plane[f_inliers]
X, colors, on_plane = X[in_front], colors[in_front], on_plane[in_front]
lim = np.percentile(X[:, 2], 98)
keep = X[:, 2] < lim
fig = plt.figure(figsize=(20, 8))
ax = fig.add_subplot(1, 2, 1)
ax.scatter(X[keep, 0], X[keep, 2], c=colors[keep], s=8)
ax.scatter(X[keep & ~on_plane, 0], X[keep & ~on_plane, 2], facecolors='none', edgecolors='red', s=40, label='off the façade plane')
ax.plot(0, 0, 'k^', markersize=12, label='camera 1')
c2 = -R.T @ t.ravel()
ax.plot(c2[0], c2[2], 'b^', markersize=12, label='camera 2')
ax.set_xlabel('x')
ax.set_ylabel('z (depth)')
ax.set_aspect('equal')
ax.legend()
ax.set_title('Top view: the façade is a line, trees and lamps are in front of it')
ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(X[keep, 0], X[keep, 2], -X[keep, 1], c=colors[keep], s=4)
ax.view_init(elev=20, azim=-60)
ax.set_xlabel('x')
ax.set_ylabel('z')
ax.set_title('Triangulated points (up to scale)')
savefig('mvg_triangulation')
plt.show()